# Exercise 2.2 — The $2 \times 2$ GOE and Linear Level Repulsion

**Chapter 2: Random Matrix Theory** &nbsp;|&nbsp; *Section 2.3: The Gaussian Orthogonal Ensemble*

---

## Motivation

The **Wigner surmise** — the nearest-neighbour spacing distribution of an integrable or chaotic quantum system — is one of the most celebrated results in random matrix theory.  Remarkably, the exact distribution for $2 \times 2$ random matrices provides an excellent approximation even for large matrices.  This exercise derives the GOE spacing distribution from first principles, demonstrating the **linear** level repulsion characteristic of systems with time-reversal symmetry ($\beta = 1$).

## Exercise Statement

Consider a $2 \times 2$ GOE matrix $H = \begin{pmatrix} a & c \\ c & b \end{pmatrix}$ with GOE measure $P(H) \propto \exp\bigl(-\mathrm{Tr}(H^2)/2\bigr)$.

**(a)** Write down the joint distribution $P(a, b, c)$.

**(b)** Find the energy spacing $s = \lambda_+ - \lambda_-$ in terms of $a, b, c$.

**(c)** Change variables to $t = a+b$, $x = a-b$, $y = 2c$.  Integrate out the trace and the angular variable to derive $P(s) \propto s\, e^{-s^2/4}$.

## Solution

### Part (a): Joint distribution

The trace of $H^2$ is:

$$
\mathrm{Tr}(H^2) = a^2 + b^2 + 2c^2.
$$

(The factor 2 on $c^2$ arises because $c$ appears both above and below the diagonal.)  The GOE measure is:

$$
P(a, b, c) \propto \exp\!\left(-\frac{a^2 + b^2 + 2c^2}{2}\right).
$$

This means $a, b \sim \mathcal{N}(0, 1)$ independently, and $c \sim \mathcal{N}(0, 1/2)$ (the off-diagonal has half the variance).

### Part (b): Spacing formula

The eigenvalues of the $2 \times 2$ symmetric matrix are:

$$
\lambda_\pm = \frac{a + b}{2} \pm \sqrt{\frac{(a-b)^2}{4} + c^2}.
$$

The spacing is:

$$
s = \lambda_+ - \lambda_- = 2\sqrt{\frac{(a-b)^2}{4} + c^2} = \sqrt{(a-b)^2 + 4c^2}.
$$

### Part (c): Deriving the Wigner surmise

**Step 1: Change variables.**  Let $t = a + b$, $x = a - b$, $y = 2c$.  Then $a = (t+x)/2$, $b = (t-x)/2$, $c = y/2$, and:

$$
a^2 + b^2 + 2c^2 = \frac{t^2 + x^2}{2} + \frac{y^2}{2} = \frac{t^2 + x^2 + y^2}{2}.
$$

The Jacobian is $|\partial(a,b,c)/\partial(t,x,y)| = 1/4$.  The distribution becomes:

$$
P(t, x, y) \propto \exp\!\left(-\frac{t^2}{4}\right) \cdot \exp\!\left(-\frac{x^2 + y^2}{4}\right).
$$

**Step 2: Integrate out the trace.** The variable $t$ (the trace $\mathrm{Tr}(H)$) decouples completely and integrates to a constant $\int e^{-t^2/4}\,dt = 2\sqrt{\pi}$.

**Step 3: Switch to polar coordinates.** The spacing is $s = \sqrt{x^2 + y^2}$.  In polar coordinates $(s, \theta)$ with $x = s\cos\theta$, $y = s\sin\theta$:

$$
dx\,dy = s\,ds\,d\theta.
$$

Integrating over the angle $\theta \in [0, 2\pi)$ gives a factor $2\pi$:

$$
\boxed{P(s) = \frac{s}{2}\,e^{-s^2/4}, \qquad s \geq 0.}
$$

(Normalization: $\int_0^\infty \frac{s}{2} e^{-s^2/4}\,ds = 1$, verified by substitution $u = s^2/4$.)

**Physical interpretation:** The factor $s$ in $P(s) \propto s$ as $s \to 0$ is the **linear level repulsion** of the GOE ($\beta = 1$).  Compare: the GUE has $P(s) \propto s^2$ (quadratic, as shown in Exercise 2.3), and the Poisson ensemble (no correlations) has $P(s) \propto \text{const}$ (no repulsion).

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

s = sp.Symbol('s', positive=True)

P_wigner = s/2 * sp.exp(-s**2/4)
norm = sp.integrate(P_wigner, (s, 0, sp.oo))
print(f"∫₀^∞ P(s) ds = {norm}")
assert norm == 1

# Mean spacing
mean_s = sp.integrate(s * P_wigner, (s, 0, sp.oo))
print(f"⟨s⟩ = {mean_s} = {sp.nsimplify(mean_s)}")

# Mode (most probable spacing): dP/ds = 0
dP = sp.diff(P_wigner, s)
mode = sp.solve(dP, s)
print(f"Mode: s* = {mode} = √2 = {sp.sqrt(2)}")

---
## Numerical Verification

In [ ]:
import numpy as np

np.random.seed(42)
n_samples = 100000

# Sample 2x2 GOE and compute spacings
spacings = []
for _ in range(n_samples):
    a, b = np.random.randn(2)
    c = np.random.randn() / np.sqrt(2)
    s = np.sqrt((a-b)**2 + 4*c**2)
    spacings.append(s)

spacings = np.array(spacings)

# Compare histogram to Wigner surmise
bins = np.linspace(0, 6, 60)
centers = (bins[:-1] + bins[1:]) / 2
hist, _ = np.histogram(spacings, bins=bins, density=True)
P_theory = centers/2 * np.exp(-centers**2/4)

residual = np.max(np.abs(hist - P_theory))
print(f"Max |histogram − P(s)| = {residual:.4f}")
assert residual < 0.02

# Verify P(0) → 0: fraction with s < 0.1
frac_small = np.mean(spacings < 0.1)
print(f"Fraction with s < 0.1: {frac_small:.4f} (linear repulsion)")

print(f"⟨s⟩ = {np.mean(spacings):.4f} (theory: √π ≈ {np.sqrt(np.pi):.4f})")
print("\nWigner surmise (GOE) confirmed. ✓")